In [1]:
# --- Celda de arranque ---
import pandas as pd
import numpy as np
from pathlib import Path

def encontrar_raiz_proyecto(marcador="requirements.txt"):
    actual = Path.cwd()
    for carpeta in [actual, *actual.parents]:
        if (carpeta / marcador).exists():
            return carpeta
    raise FileNotFoundError(f"No encontré '{marcador}' subiendo desde {actual}")

RAIZ = encontrar_raiz_proyecto()
print("Raíz del proyecto:", RAIZ)

Raíz del proyecto: C:\Users\mgdbj\xm-spot-price-predictor


In [2]:
# --- Celda 1: Cargar train (2019-2025) y test (2026), unir en una sola serie ---
df_train = pd.read_csv(RAIZ / "data" / "processed" / "dataset_maestro_2019_2025.csv", parse_dates=["fecha_hora"])
df_test = pd.read_csv(RAIZ / "data" / "processed" / "dataset_maestro_2026.csv", parse_dates=["fecha_hora"])

df_train["conjunto"] = "train"
df_test["conjunto"] = "test"

df_completo = pd.concat([df_train, df_test], ignore_index=True).sort_values("fecha_hora").reset_index(drop=True)
print(df_completo.shape)

(66576, 9)

In [3]:
# --- Celda 2 (CORREGIDA): Rezagos, respetando el horizonte de 24-72h ---
for horas in [24, 168]:
    df_completo[f"precio_lag{horas}h"] = df_completo["precio_bolsa"].shift(horas)

df_completo[["fecha_hora", "precio_bolsa", "precio_lag24h", "precio_lag168h"]].head(3)

,fecha_hora,precio_bolsa,precio_lag24h,precio_lag168h
0,2019-01-01 00:00:00,318.4231,NaN,NaN
1,2019-01-01 01:00:00,318.4231,NaN,NaN
2,2019-01-01 02:00:00,318.4231,NaN,NaN


In [4]:
# --- Celda 3 (CORREGIDA): Promedios móviles, mismo cuidado de horizonte ---
df_completo["precio_media_24h"] = df_completo["precio_bolsa"].shift(24).rolling(window=24).mean()
df_completo["precio_media_7d"] = df_completo["precio_bolsa"].shift(24).rolling(window=24 * 7).mean()
df_completo["precio_media_30d"] = df_completo["precio_bolsa"].shift(24).rolling(window=24 * 30).mean()

df_completo[["fecha_hora", "precio_bolsa", "precio_media_24h", "precio_media_7d", "precio_media_30d"]].tail(3)

,fecha_hora,precio_bolsa,precio_media_24h,precio_media_7d,precio_media_30d
66573,2026-08-05 21:00:00,1060.0,998.513900,910.017780,811.086334
66574,2026-08-05 22:00:00,1049.5,998.680857,912.101170,811.399820
66575,2026-08-05 23:00:00,1038.5,999.889480,914.303608,811.773029


In [5]:
# --- Celda 4: Armónicos de calendario (Fourier de hora del día y día de la semana) ---
# Esto es, literalmente, el periodograma del Paso 1.5 convertido en insumo del
# modelo -- deja de ser solo diagnóstico. El problema que resuelve: para
# XGBoost, la hora 23 y la hora 0 son números lejanos (23 vs 0), aunque en la
# realidad son consecutivas. Seno/coseno "envuelven" el ciclo para que el
# modelo vea esa continuidad. Un solo armónico por ciclo alcanza -- a
# diferencia de Prophet, los árboles ya son capaces de aprender formas no
# lineales por sí solos, así que esto resuelve la envoltura, no añade
# expresividad extra.

df_completo["hora"] = df_completo["fecha_hora"].dt.hour
df_completo["hora_sin"] = np.sin(2 * np.pi * df_completo["hora"] / 24)
df_completo["hora_cos"] = np.cos(2 * np.pi * df_completo["hora"] / 24)

df_completo["dia_semana"] = df_completo["fecha_hora"].dt.dayofweek
df_completo["dia_semana_sin"] = np.sin(2 * np.pi * df_completo["dia_semana"] / 7)
df_completo["dia_semana_cos"] = np.cos(2 * np.pi * df_completo["dia_semana"] / 7)

# Incluido también el anual, pero recuerda el hallazgo del 1.5: es un
# componente débil y difuso, no una aguja como los dos anteriores.
df_completo["dia_anio"] = df_completo["fecha_hora"].dt.dayofyear
df_completo["dia_anio_sin"] = np.sin(2 * np.pi * df_completo["dia_anio"] / 365.25)
df_completo["dia_anio_cos"] = np.cos(2 * np.pi * df_completo["dia_anio"] / 365.25)

print("Columnas de calendario listas.")

Columnas de calendario listas.


In [6]:
# --- Variables hidrológicas derivadas (CORREGIDA: también sobrescribe la cruda) ---
H = 24  # horizonte base

for col in ["volumen_embalses", "aportes_hidricos"]:
    base = df_completo[col].shift(H)
    df_completo[col] = base  # <-- la corrección: la columna cruda también queda rezagada
    df_completo[f"{col}_delta_1d"] = base.diff(24)
    df_completo[f"{col}_delta_7d"] = base.diff(24 * 7)
    df_completo[f"{col}_media_7d"] = base.rolling(24 * 7).mean()
    df_completo[f"{col}_media_30d"] = base.rolling(24 * 30).mean()
    df_completo[f"{col}_vs_media30d"] = base / base.rolling(24 * 30).mean()

print("Variables hidrológicas derivadas listas -- columna cruda también corregida por horizonte.")

Variables hidrológicas derivadas listas -- columna cruda también corregida por horizonte.


In [7]:
print([c for c in df_completo.columns if "media_30d" in c])

['precio_media_30d', 'volumen_embalses_media_30d', 'aportes_hidricos_media_30d']


In [8]:
# --- Última ronda: volatilidad/régimen + demanda rezagada ---
# Todo con shift(H) primero -- mismo horizonte que el resto de variables,
# para no repetir la fuga de precio_lag1h de hace un rato.

H = 24  # horizonte base, igual que el resto del pipeline

precio_base = df_completo["precio_bolsa"].shift(H)

df_completo["precio_std_24h"] = precio_base.rolling(24).std()
df_completo["precio_std_7d"] = precio_base.rolling(24 * 7).std()
df_completo["precio_rango_24h"] = precio_base.rolling(24).max() - precio_base.rolling(24).min()
# Ratio > 1 = más volátil que lo típico reciente; < 1 = más tranquilo de lo normal
df_completo["ratio_volatilidad"] = df_completo["precio_std_24h"] / df_completo["precio_std_7d"]

demanda_base = df_completo["demanda"].shift(H)
df_completo["demanda_lag24h"] = demanda_base
df_completo["demanda_lag48h"] = df_completo["demanda"].shift(H + 24)
df_completo["demanda_lag72h"] = df_completo["demanda"].shift(H + 48)
df_completo["demanda_media_24h"] = demanda_base.rolling(24).mean()

print("Variables de volatilidad y demanda listas.")

Variables de volatilidad y demanda listas.


In [9]:
# --- Celda 5 (versión final, consolidada): Separar en train/test y guardar ---
columnas_no_nulas = [
    "precio_lag24h", "precio_lag168h", "precio_media_30d",
    "volumen_embalses_media_30d", "aportes_hidricos_media_30d",
    "precio_std_7d", "demanda_lag72h",
]

df_train_features = df_completo[df_completo["conjunto"] == "train"].drop(columns=["conjunto"]).dropna(
    subset=columnas_no_nulas
).reset_index(drop=True)

df_test_features = df_completo[df_completo["conjunto"] == "test"].drop(columns=["conjunto"]).reset_index(drop=True)

print("Train con features:", df_train_features.shape)
print("Test con features:", df_test_features.shape)
print("Nulos en train:", df_train_features.isna().sum().sum())
print("Nulos en test:", df_test_features.isna().sum().sum())

df_train_features.to_csv(RAIZ / "data" / "processed" / "dataset_features_2019_2025.csv", index=False)
df_test_features.to_csv(RAIZ / "data" / "processed" / "dataset_features_2026.csv", index=False)
print("\nGuardado: dataset_features_2019_2025.csv y dataset_features_2026.csv")

Train con features:

 (60625, 40)
Test con features: (5208, 40)


Nulos en train: 0
Nulos en test: 0



Guardado: dataset_features_2019_2025.csv y dataset_features_2026.csv
